# OULAD Early-Risk Classification

Primary cutoff: 28 days. Secondary experiment: 56 days. Target: `Fail`/`Withdrawn` = 1, `Pass`/`Distinction` = 0.

The split is grouped by `id_student` so the same learner cannot appear in train and test.

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib joblib tensorflow
import os, zipfile, numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import *
from src.feature_engineering import build_dataset
from src.models import make_preprocessor, traditional_models, make_pipeline
from src.deep_learning import make_tf_datasets, build_mlp, train_mlp


In [ ]:
ZIP_PATH='/content/anonymisedData.zip'; DATA_DIR='/content/data'
if os.path.exists(ZIP_PATH) and not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR,exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DATA_DIR)
df=build_dataset(DATA_DIR,28)
print(df.shape, df.at_risk.value_counts(normalize=True))


In [ ]:
def split(df):
    X=df.drop(columns=['at_risk','id_student']); y=df.at_risk.astype(int); g=df.id_student
    a,b=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(X,y,g))
    Xtr,Xte=X.iloc[a],X.iloc[b]; ytr,yte=y.iloc[a],y.iloc[b]; gt=g.iloc[a]
    c,d=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=43).split(Xtr,ytr,gt))
    return Xtr.iloc[c],Xtr.iloc[d],Xte,ytr.iloc[c],ytr.iloc[d],yte
Xtr,Xval,Xtest,ytr,yval,ytest=split(df)
for name,model in traditional_models().items():
    pipe=make_pipeline(Xtr,model); pipe.fit(Xtr,ytr); p=pipe.predict_proba(Xtest)[:,1]
    print(name, 'AUC=',round(roc_auc_score(ytest,p),4),'F1=',round(f1_score(ytest,p>=.5),4))


## TensorFlow/Keras experiment

Run this section in a TensorFlow environment such as Google Colab. The neural network uses `tf.data`, early stopping and a small hyperparameter comparison.

In [ ]:
prep=make_preprocessor(Xtr)
A=prep.fit_transform(Xtr).astype('float32'); B=prep.transform(Xval).astype('float32'); C=prep.transform(Xtest).astype('float32')
train_ds,val_ds=make_tf_datasets(A,ytr.to_numpy(),B,yval.to_numpy(),128)
configs=[((128,64,32),.30,1e-3),((128,64),.20,1e-3),((256,128,64),.35,5e-4)]
best=None; best_auc=-1
for units,dropout,lr in configs:
    m=build_mlp(A.shape[1],units,dropout,lr); h=train_mlp(m,train_ds,val_ds,80)
    auc=max(h.history['val_roc_auc'])
    if auc>best_auc: best_auc=auc; best=m; best_history=h.history
p=best.predict(C,batch_size=256).ravel()
print('DL ROC-AUC:',roc_auc_score(ytest,p),'F1:',f1_score(ytest,p>=.5),'Recall:',recall_score(ytest,p>=.5))
best.save('best_deep_learning_model.keras')


In [ ]:
# 56-day comparison
df56=build_dataset(DATA_DIR,56)
Xtr,Xval,Xtest,ytr,yval,ytest=split(df56)
prep=make_preprocessor(Xtr); A=prep.fit_transform(Xtr); C=prep.transform(Xtest)
m=traditional_models()['HistGradientBoosting']; m.fit(A,ytr); p=m.predict_proba(C)[:,1]
print('56-day HGB:',accuracy_score(ytest,p>=.5),precision_score(ytest,p>=.5),recall_score(ytest,p>=.5),f1_score(ytest,p>=.5),roc_auc_score(ytest,p))


Save the DL `.keras` model and the best traditional `.joblib` model to Google Drive. Do not commit the raw OULAD CSVs or model binaries to GitHub.